In [ ]:
import pandas as pd
import datetime as dt
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
def tbillrate():
    df = pd.read_csv("data/TB3MS.csv")
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df["TB3MS"] /= 100
    return df.set_index("observation_date")
def compute_gold_tr(df):
    for colname in df.columns:
        df[colname+'TR'] = df[colname]/df[colname].shift(1)
    return df
def goldprice(col):
    df = pd.read_csv("data/cmo-data-monthly.csv")
    print(df.columns)
    df = df[["date"]+list(col.keys())]
    df["date"] = pd.to_datetime(df["date"])+dt.timedelta(days=1)
    df = df.loc[df["date"]>=dt.datetime(1971,2,1)]
    return compute_gold_tr(df.set_index("date").rename(columns=col))
def read_shiller_out(col):
    df = pd.read_csv("data/shiller_out.csv")
    df['Date'] = pd.to_datetime(df['Date'], format="%Y-%m-%d")+dt.timedelta(days=-14)
    df = df.set_index("Date")
    #for c in df.columns:
    #    df[c] = df[c].astype(float)
    #df['Rate10y'] = df['Rate10y']/100
    return df.join(tbillrate()).join(goldprice(col),how="inner")

"""
def compute_bond_tr(df):
    r = df['Rate10y']
    T = 10
    duration = -(1-np.exp(-r*T))/r
    bondcarry = r.shift(1)/12
    bondtotalret = 1+(r-r.shift(1))*duration+bondcarry
    df["bondTR"] = bondtotalret
def compute_eq_tr(df):
    df['eqTR'] = df['SP500']/df['SP500'].shift(1)+df['Div']/12/df['SP500']
def compute_cpi_tr(df):
    df['cpiTR'] = 1+df['CPI'].pct_change()
    """
def compute_tbill_tr(df):
    df['tbTR'] = 1+df['TB3MS'].shift(1)*365/360/12

def metrics_monthly_ret(df):
    trcolumns = [c for c in df.columns if "TR"==c[-2:] and c!="cpiTR"]
    logret = np.log(df[trcolumns].dropna())
    mu,sigma = logret.mean()*12,logret.std()*np.sqrt(12)
    sigma["tbTR"] = 0
    mu += 0.5*sigma**2
    r = mu["tbTR"]
    riskycol = trcolumns[:-1]
    mustar = np.expm1(mu[riskycol]-r)
    data = {'mustar':mustar,'sigma':sigma[riskycol]}
    data['sharpe'] = data['mustar']/data['sigma']
    corr = logret[riskycol].corr()
    cov = logret[riskycol].cov()*12
    wstar = np.linalg.solve(cov,mustar)
    K = np.sum(wstar)
    data['w'] = wstar/K
    for c in riskycol:
        data[f'rho({c[:-2]})'] = corr[c]
    return pd.DataFrame(data,index=riskycol),r,K

def show_returns(df,filename):
    dfmetrics,r,K = metrics_monthly_ret(df)
    print(dfmetrics.index)
    for c in dfmetrics.index:
        plt.plot(np.cumprod(df[c]),
                label=f"{c[:-2]}: $\mu^*$={dfmetrics.loc[c,'mustar']:.1%}, $\sigma$={dfmetrics.loc[c,'sigma']:.0%}, S={dfmetrics.loc[c,'sharpe']:.2f}, w={dfmetrics.loc[c,'w']:.0%}")
    plt.legend()
    plt.title(f"Asset Total Return r={r:.1%} K={K:.1f}")
    plt.ylabel("log total return")
    plt.yscale('log')  
    plt.xlabel(f"from {str(df.index[0])[:10]} to {str(df.index[-1])[:10]}")  
    plt.grid(True)  
    if not filename is None:
        plt.savefig(filename)
        plt.close()
    else:
        plt.show()
    return dfmetrics

def show_col_returns(col,show=True):
    df = read_shiller_out(col)
    compute_tbill_tr(df)
    assets = [c[:-2] for c in df.columns if "TR" in c and c not in ["cpiTR","tbTR"]]
    filename = None if show else "_".join(assets).replace(" (no roll)","noroll")+".png"
    dfmetrics = show_returns(df.loc[df.index>=dt.datetime(1960,1,1)],filename)
    return dfmetrics,filename


In [ ]:
col = {"CRUDE_DUBAI":"crude (no roll)", "GOLD":"gold"}
dfmetrics,filename = show_col_returns(col,show=True)
dfmetrics.to_markdown(),filename


In [ ]:
col = {"CRUDE_DUBAI":"crude", "GOLD":"gold","COPPER":"copper"}
dfmetrics,filename = show_col_returns(col)
dfmetrics.to_markdown()


In [ ]:
dfcl = pd.read_csv("data/cl.csv")
dfcl["date"] = pd.to_datetime(dfcl["date"])
dfcl["backwardTR"] = pd.NA
dfcl["oilTR"] = pd.NA
dfcl = dfcl.set_index("date")
dfcl = dfcl.resample('M').last()
dfcl = dfcl.reset_index()
dfcl["date"] = dfcl["date"]+dt.timedelta(days=1)
dfcl = dfcl.set_index("date")
df = read_shiller_out({'GOLD': 'gold', 'COPPER': 'copper'})
dfcl = dfcl.join(df)
compute_tbill_tr(dfcl)
contango = (1+(dfcl["CL6"]-dfcl["CL1"])/dfcl["CL6"])**(1/5)-1-dfcl["TB3MS"]/12
totret   = dfcl["CL6"].pct_change()-contango.shift(1)+1#+(dfcl["tbTR"]-1)
dfcl["backwardTR"] = 1+(totret-1)*np.where(contango<0.0088,1,-1)
dfcl["oilTR"] = totret
dfcl["cumbackwardationpnl"] = np.cumprod(dfcl["backwardTR"])
dfcl["contango"] = contango
# for c in ["CL1","CL6"]:
#     plt.plot(dfcl[c],label=c)
# plt.legend()
# plt.show()
# plt.plot(contango,label="contango")
# plt.title("contango rate")
# plt.axhline(y=0,color="black")
# plt.legend()
# plt.show()
dfcl

In [ ]:
plt.plot(dfcl["cumbackwardationpnl"],label="backwardation")
# plt.plot(dfcl.loc[dfcl["backwardTR"]==1].index, 
#          dfcl.loc[dfcl["backwardTR"]==1, "cumbackwardationpnl"], 
#          color="red", 
#          linestyle='none',     # <-- this removes the line
#          marker='o',           # circle marker
#          markersize=1,
#          label="contango")
plt.plot(np.cumprod(dfcl["oilTR"]),label="total return")
plt.yscale('log')
plt.legend()
plt.grid(True)

In [ ]:
metrics_monthly_ret(dfcl)

In [ ]:
x = dfcl["TB3MS"]
y = dfcl["Rate10y"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("3M tbill rate")
plt.ylabel("10y rate")
plt.title("10y rate vs 3m rate")
plt.show()
x = (dfcl["TB3MS"]).shift(1)
y = dfcl["bondTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("3m rate")
plt.ylabel("bond net total return")
plt.title("bond net return vs 3m rate")
plt.show()

In [ ]:
x = (dfcl["Rate10y"]).shift(1)
y = dfcl["goldTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
print(f"pivot: {-b/a:.2%}")
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("10y rate")
plt.ylabel("gold net total return")
plt.title("Gold net return vs 10y rate")
plt.show()

In [ ]:
x = (dfcl["Rate10y"]-dfcl["TB3MS"]).shift(1)
y = dfcl["bondTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
print(f"pivot: {-b/a:.2%}")
errstd = np.std(y_clean - y_pred)
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("10y - 3m rate spread")
plt.ylabel("bond net total return")
plt.title("bond net return vs rate spread")
plt.show()

In [ ]:
# x = (dfcl["Rate10y"]-dfcl["TB3MS"]).shift(1)
# y = dfcl["eqTR"]-dfcl["tbTR"]
# mask = ~(np.isnan(x) | np.isnan(y))
# x_clean = x[mask].values
# y_clean = y[mask].values
# coeffs = np.polyfit(x_clean, y_clean, 1)
# a, b = coeffs
# y_pred = a * x_clean + b
# print(f"pivot: {-b/a:.2%}")
# errstd = np.std(y_clean - y_pred)
# plt.scatter(x, y, alpha=0.2)
# plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
# plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
#          transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
# plt.xlabel("10y - 3m rate spread")
# plt.ylabel("eq net total return")
# plt.title("eq net return vs rate spread")
# plt.show()

In [ ]:
x = (dfcl["contango"]).shift(1)
y = totret-1
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
print(f"pivot: {-b/a:.2%}")
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("oil contango")
plt.ylabel("future total return")
plt.title("oil fut return vs contango")
plt.show()

In [ ]:
x = (dfcl["eqTR"]-dfcl["tbTR"]).rolling(8).mean().shift(1)
y = dfcl["eqTR"]-dfcl["tbTR"]
mask = ~(np.isnan(x) | np.isnan(y))
x_clean = x[mask].values
y_clean = y[mask].values
coeffs = np.polyfit(x_clean, y_clean, 1)
a, b = coeffs
y_pred = a * x_clean + b
errstd = np.std(y_clean - y_pred)
print(f"pivot: {-b/a:.2%}")
plt.scatter(x, y, alpha=0.2)
plt.plot(x_clean, np.polyval(coeffs, x_clean), 'r-', lw=2)
plt.text(0.05, 0.9, f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f} + {errstd:.4f} e', 
         transform=plt.gca().transAxes, bbox=dict(facecolor='white', alpha=0.7))
plt.xlabel("past return")
plt.ylabel("eq return")
plt.title("eq retun vs past return")
plt.show()